# 🔗 Module 8.3: Pipeline Tracking

**Goal**: Track a complete ML pipeline — from data preprocessing to evaluation — using MLFlow's tracking API with structured tags.

---

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import json
import time

mlflow.autolog(disable=True)
mlflow.set_experiment("08_pipeline_tracking")

print("✅ Setup complete!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## Pipeline Architecture

```
Parent Run: "Full ML Pipeline"
    ├── Stage 1: Data Preparation      (nested run)
    ├── Stage 2: Feature Engineering    (nested run)
    ├── Stage 3: Model Training         (nested run)
    └── Stage 4: Evaluation             (nested run)
```

Each stage logs its own parameters, metrics, and artifacts, all linked under a parent run.

## Stage 1: Data Preparation

In [ ]:
with mlflow.start_run(run_name="full_ml_pipeline") as parent_run:
    mlflow.set_tag("pipeline_version", "1.0")
    mlflow.set_tag("pipeline_type", "classification")
    
    pipeline_id = parent_run.info.run_id
    
    # ===== STAGE 1: DATA PREPARATION =====
    with mlflow.start_run(run_name="stage_1_data_prep", nested=True):
        mlflow.set_tag("stage", "data_preparation")
        mlflow.set_tag("stage_number", "1")
        
        # Load data
        wine = load_wine()
        X = pd.DataFrame(wine.data, columns=wine.feature_names)
        y = pd.Series(wine.target, name="target")
        
        # Log dataset info
        mlflow.log_params({
            "dataset": "wine",
            "n_samples": len(X),
            "n_features": X.shape[1],
            "n_classes": len(np.unique(y)),
            "test_size": 0.2,
        })
        
        # Split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        mlflow.log_metrics({
            "train_samples": len(X_train),
            "test_samples": len(X_test),
        })
        
        # Log data sample as artifact
        sample = X_train.head(10).copy()
        sample["target"] = y_train.head(10).values
        mlflow.log_text(sample.to_csv(index=False), "data_sample.csv")
        
        # Log class distribution
        class_dist = y_train.value_counts().to_dict()
        mlflow.log_dict({str(k): int(v) for k, v in class_dist.items()}, "class_distribution.json")
        
        print("  ✅ Stage 1: Data Preparation complete")
        print(f"     Train: {len(X_train)}, Test: {len(X_test)}")
    
    # ===== STAGE 2: FEATURE ENGINEERING =====
    with mlflow.start_run(run_name="stage_2_feature_eng", nested=True):
        mlflow.set_tag("stage", "feature_engineering")
        mlflow.set_tag("stage_number", "2")
        
        # Standard scaling
        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(
            scaler.fit_transform(X_train), 
            columns=X_train.columns, 
            index=X_train.index
        )
        X_test_scaled = pd.DataFrame(
            scaler.transform(X_test), 
            columns=X_test.columns, 
            index=X_test.index
        )
        
        mlflow.log_params({
            "scaling_method": "StandardScaler",
            "n_features_in": X_train_scaled.shape[1],
            "n_features_out": X_train_scaled.shape[1],
        })
        
        # Log feature stats
        feature_stats = X_train_scaled.describe().T[["mean", "std"]].to_dict()
        mlflow.log_dict(feature_stats, "feature_statistics.json")
        
        print("  ✅ Stage 2: Feature Engineering complete")
        print(f"     Features: {X_train_scaled.shape[1]} (scaled)")
    
    # ===== STAGE 3: MODEL TRAINING =====
    with mlflow.start_run(run_name="stage_3_training", nested=True):
        mlflow.set_tag("stage", "model_training")
        mlflow.set_tag("stage_number", "3")
        
        model_params = {
            "model_type": "RandomForest",
            "n_estimators": 150,
            "max_depth": 8,
            "min_samples_split": 5,
            "random_state": 42,
        }
        mlflow.log_params(model_params)
        
        start_time = time.time()
        model = RandomForestClassifier(
            n_estimators=model_params["n_estimators"],
            max_depth=model_params["max_depth"],
            min_samples_split=model_params["min_samples_split"],
            random_state=model_params["random_state"]
        )
        model.fit(X_train_scaled, y_train)
        train_time = time.time() - start_time
        
        mlflow.log_metric("training_time_seconds", train_time)
        
        # Log feature importance
        importance = dict(zip(X_train.columns, model.feature_importances_))
        mlflow.log_dict(importance, "feature_importances.json")
        
        # Log the model
        signature = infer_signature(X_train_scaled, model.predict(X_train_scaled))
        mlflow.sklearn.log_model(model, "model", signature=signature)
        
        print(f"  ✅ Stage 3: Model Training complete ({train_time:.2f}s)")
    
    # ===== STAGE 4: EVALUATION =====
    with mlflow.start_run(run_name="stage_4_evaluation", nested=True):
        mlflow.set_tag("stage", "evaluation")
        mlflow.set_tag("stage_number", "4")
        
        y_pred = model.predict(X_test_scaled)
        
        metrics = {
            "test_accuracy": accuracy_score(y_test, y_pred),
            "test_f1_weighted": f1_score(y_test, y_pred, average='weighted'),
        }
        mlflow.log_metrics(metrics)
        
        # Log classification report
        report = classification_report(y_test, y_pred, target_names=wine.target_names)
        mlflow.log_text(report, "classification_report.txt")
        
        # Log confusion matrix as plot
        import matplotlib.pyplot as plt
        from sklearn.metrics import ConfusionMatrixDisplay
        
        fig, ax = plt.subplots(figsize=(8, 6))
        ConfusionMatrixDisplay.from_predictions(
            y_test, y_pred, display_labels=wine.target_names, ax=ax, cmap='Blues'
        )
        plt.tight_layout()
        fig.savefig("confusion_matrix.png", dpi=150)
        mlflow.log_artifact("confusion_matrix.png", "plots")
        plt.show()
        import os; os.remove("confusion_matrix.png")
        
        print(f"  ✅ Stage 4: Evaluation complete")
        for k, v in metrics.items():
            print(f"     {k}: {v:.4f}")
    
    # Log summary on parent
    mlflow.log_metrics(metrics)  # Best metrics on parent
    mlflow.set_tag("pipeline_status", "completed")
    
    print(f"\n🎉 Pipeline complete! Parent run ID: {pipeline_id}")

## Query Pipeline Stages

In [ ]:
# Find all stages of the pipeline
stages = mlflow.search_runs(
    experiment_names=["08_pipeline_tracking"],
    filter_string=f"tags.mlflow.parentRunId = '{pipeline_id}'",
    order_by=["tags.stage_number ASC"]
)

print("📊 Pipeline Stages:")
stage_cols = [c for c in ["tags.mlflow.runName", "tags.stage", "status"] if c in stages.columns]
if stage_cols:
    print(stages[stage_cols].to_string(index=False))

## 🔑 Key Takeaways

1. Use **nested runs** to structure pipeline stages under a parent
2. **Tag each stage** (`stage=data_preparation`, `stage_number=1`) for easy querying
3. Each stage logs its **own params, metrics, and artifacts**
4. The **parent run** holds summary metrics and pipeline metadata
5. Query stages with `tags.mlflow.parentRunId` filter

---

## ➡️ Next: `04_database_backends.ipynb`